# Camada Ouro - Agregações e Indicadores de Churn

Tabelas agregadas a partir da camada Silver pra análise de negócio e ML.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
SILVER_INPUT = Path("../silver/telco_churn_silver.parquet")
GOLD_DIR = Path("../gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)
ARQUIVO_FEATURES = GOLD_DIR / "customer_features.parquet"
ARQUIVO_CHURN_REF = GOLD_DIR / "churn_reference.parquet"
ARQUIVO_SUMMARY = GOLD_DIR / "churn_summary.parquet"

In [11]:
df = pd.read_parquet(SILVER_INPUT)

In [12]:
FEATURES_ML = [
    # Dados demográficos
    "gender",
    "SeniorCitizen",
    "is_single_no_dependents",
    "family_size_norm",

    # Serviços de telefonia
    "PhoneService",
    "MultipleLines",

    # Perfil de internet
    "internet_service_flag",
    "internet_dsl_flag",
    "internet_fiber_flag",

    # Add-ons de internet
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "support_services_count_norm",
    "streaming_services_count_norm",
    "service_count_norm",

    # Contrato e pagamento
    "is_month_to_month",
    "contract_term_months_norm",
    "PaperlessBilling",
    "payment_automatic",
    "payment_electronic_check",

    # Financeiro
    "tenure_norm",
    "MonthlyCharges_norm",
    "TotalCharges_norm",
    "charges_per_month_ratio_norm",
]

In [13]:
df_features = df[FEATURES_ML].copy()
assert df_features.isnull().sum().sum() == 0, "Nulos encontrados no customer_features!"
assert df_features.select_dtypes(include="object").empty, "Strings encontradas no customer_features!"

print(f"customer_features: {df_features.shape[0]:,} linhas × {df_features.shape[1]} colunas")
print("\nFeatures selecionadas para clustering:")
print(df_features.dtypes)
print(f"\nEstatísticas descritivas:")
print(df_features.describe().round(3))
df_features.to_parquet(ARQUIVO_FEATURES, index=False, engine="pyarrow")

customer_features: 7,032 linhas × 27 colunas

Features selecionadas para clustering:
gender                             int64
SeniorCitizen                      int64
is_single_no_dependents            int64
family_size_norm                 float64
PhoneService                       int64
MultipleLines                      int64
internet_service_flag              int64
internet_dsl_flag                  int64
internet_fiber_flag                int64
OnlineSecurity                     int64
OnlineBackup                       int64
DeviceProtection                   int64
TechSupport                        int64
StreamingTV                        int64
StreamingMovies                    int64
support_services_count_norm      float64
streaming_services_count_norm    float64
service_count_norm               float64
is_month_to_month                  int64
contract_term_months_norm        float64
PaperlessBilling                   int64
payment_automatic                  int64
payment_elect

In [14]:
df_ref = df[["customerID", "Churn"]].copy()
print(f"churn_reference: {df_ref.shape[0]:,} linhas × {df_ref.shape[1]} colunas")
print(f"\nDistribuição de Churn na referência:")
print(df_ref["Churn"].value_counts())
print(f"Taxa geral de churn: {df_ref['Churn'].mean():.1%}")
df_ref.to_parquet(ARQUIVO_CHURN_REF, index=False, engine="pyarrow")

churn_reference: 7,032 linhas × 2 colunas

Distribuição de Churn na referência:
Churn
0    5163
1    1869
Name: count, dtype: int64
Taxa geral de churn: 26.6%


In [15]:
summary = df.groupby(["tenure_group", "Contract"]).agg(
    taxa_churn=("Churn", "mean"),
    total_clientes=("Churn", "count"),
    media_cobranca_mensal=("MonthlyCharges", "mean"),
    proporcao_fibra=("internet_fiber_flag", "mean"),
    proporcao_mes_a_mes=("is_month_to_month", "mean"),
    media_servicos=("service_count_norm", "mean"),
).round(3).reset_index()

print(f"churn_summary: {summary.shape}")
print("\nResumo por tenure_group e Contract:")
print(summary.to_string(index=False))
summary.to_parquet(ARQUIVO_SUMMARY, index=False, engine="pyarrow")

churn_summary: (12, 8)

Resumo por tenure_group e Contract:
tenure_group       Contract  taxa_churn  total_clientes  media_cobranca_mensal  proporcao_fibra  proporcao_mes_a_mes  media_servicos
       0-12m Month-to-month       0.514            1994                 58.218            0.459                  1.0           0.245
       0-12m       One year       0.106             123                 35.928            0.057                  0.0           0.172
       0-12m       Two year       0.000              58                 28.766            0.000                  0.0           0.136
      13-24m Month-to-month       0.377             737                 69.310            0.577                  1.0           0.368
      13-24m       One year       0.081             197                 44.879            0.117                  0.0           0.254
      13-24m       Two year       0.000              90                 32.307            0.011                  0.0           0.182
      25-

In [16]:
print("\nGold salva com sucesso:")
for f in [ARQUIVO_FEATURES, ARQUIVO_CHURN_REF, ARQUIVO_SUMMARY]:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


Gold salva com sucesso:
  - customer_features.parquet (199.4 KB)
  - churn_reference.parquet (91.1 KB)
  - churn_summary.parquet (5.9 KB)
